# Rung 2 — Is the optimiser the problem?

**Question.** Rung 1 returned a point. Is it *the* minimum, or just the basin the
optimiser happened to start in?

This is a question that has to be asked and is usually only assumed. `scipy` reports
`success: True` when its convergence test is satisfied, which says nothing about whether
some other starting point would have found somewhere better.

**A note on what follows.** The honest answer for this model turns out to be *no, the
optimiser is not the problem* — and that result is worth a rung precisely because it is
not what one expects to find. Knowing when you do **not** need global optimisation is as
useful as knowing when you do, and the check is cheap. What does bite on this problem is
something else, and it is what rung 3 is about.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from inferladder import default_sir_dataset, default_sir_model
from inferladder.inference import cost_surface, multistart, negative_log_likelihood, random_starts

dataset = default_sir_dataset()
model = default_sir_model()
objective = negative_log_likelihood(model, dataset.y, dataset.t)

## Multistart

Twelve starting points scattered log-uniformly around the truth by up to a factor of
about three, each run to convergence.

In [ ]:
starts = random_starts(dataset.theta_true, 12, spread=1.2, seed=1)
best, runs = multistart(objective, starts, model.param_names)

values = np.array([run.objective_value for run in runs])
reached = values < values.min() + 0.5
print(f"best objective        : {values.min():.4f}")
print(f"runs reaching the best: {reached.sum()} of {len(runs)}")
print(f"worst run             : {values.max():.4f}")
print(f"failed to converge    : {sum(not r.success for r in runs)}")

In [ ]:
estimates = np.array([run.theta for run in runs])

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(
    estimates[reached, 0],
    estimates[reached, 1],
    s=45,
    color="#33628f",
    label="converged to the best optimum",
    zorder=3,
)
if (~reached).any():
    ax.scatter(
        estimates[~reached, 0],
        estimates[~reached, 1],
        s=45,
        marker="x",
        color="#c1502e",
        label="stopped somewhere worse",
        zorder=3,
    )
ax.scatter(
    [dataset.truth["beta"]],
    [dataset.truth["gamma"]],
    s=160,
    marker="*",
    color="#3c3c3c",
    label="truth",
    zorder=4,
)
ax.set(xlabel="beta", ylabel="gamma", title="Where twelve independent searches ended up")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

Essentially all of them land on the same point. There is no second basin to fall into:
on the full epidemic curve this likelihood has one optimum, and a single local search
finds it.

The runs that *do* fail fail in a specific way — they wander into parameters the model
rejects (a reporting fraction above 1, a negative rate), where the objective is `+inf`.
That is a wall, not a local minimum, and the cure is the log-parameterisation used
throughout rather than a global optimiser.

## Looking at the surface

If there is only one minimum, why is any of this delicate? Evaluate the objective on a
grid through the optimum. This is a *slice*: the other parameters are held at their fitted
values.

In [ ]:
surface = cost_surface(objective, best.theta, (0, 1), model.param_names, factor=1.35, resolution=36)

fig, ax = plt.subplots(figsize=(6.2, 4.4))
levels = best.objective_value + np.array([0.5, 1, 2, 4, 8, 16, 32, 64, 128])
contours = ax.contour(surface.x, surface.y, surface.values, levels=levels, cmap="viridis")
ax.clabel(contours, inline=True, fontsize=7, fmt="%.0f")
ax.plot(best.theta[0], best.theta[1], "o", color="#c1502e", ms=8, label="fit")
ax.plot(dataset.truth["beta"], dataset.truth["gamma"], "*", color="#3c3c3c", ms=15, label="truth")
ax.set(
    xlabel=surface.x_name,
    ylabel=surface.y_name,
    title="Negative log-likelihood: one minimum, in a long diagonal valley",
)
ax.legend()
fig.tight_layout()

One minimum — and a valley stretched along a diagonal. Moving `beta` and `gamma`
*together* along that direction barely changes the fit.

That is the real difficulty, and it is not an optimisation difficulty. The optimiser
finds the bottom of the valley without trouble. The problem is that the bottom is nearly
flat, so *where along it* you stop is decided by noise rather than by the data.

## When to stop here

**This rung is enough to trust a point estimate as an optimisation result.** A multistart
that agrees is real evidence, and it costs twelve fits.

**It says nothing about whether the estimate is determined by the data.** A flat valley
returns a confident-looking number from every start. Quantifying that flatness — and
finding out which combinations of parameters it hides — is rung 3.